In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers
import itertools
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 1. Load, clean & map gene names

In [5]:
#reading in csv file
df = pd.read_csv('denseDataOnlyDownload-2.tsv', sep='\t')

#mapping csv file names to gene markers, then renaming columns
ensembl_map = {
    'ENSG00000082175.15': 'PGR',
    'ENSG00000141736.14': 'ERBB2',
    'ENSG00000087586.18': 'SLAMF7',
    'ENSG00000171862.11': 'PTEN',
    'ENSG00000146648.19': 'EGFR',
    'ENSG00000103855.18': 'CD276',
    'ENSG00000090339.9':  'ICAM1'
}
df = df.rename(columns=ensembl_map)
df = df.rename(columns={'sample_type.samples':'sample_type','primary_diagnosis.diagnoses':'primary_diagnosis'})
GENES = list(ensembl_map.values())

# Remove nulls
before = len(df)
df = df.dropna(subset=GENES + ['sample_type'])
print(f'Removed {before-len(df)} rows with nulls → {len(df)} samples remaining')
print(f'Sample types:\n{df["sample_type"].value_counts()}')

Removed 31 rows with nulls → 1226 samples remaining
Sample types:
sample_type
Primary Tumor          1106
Solid Tissue Normal     113
Metastatic                7
Name: count, dtype: int64


## 2. Split tumor vs normal

In [6]:
#combining Primary Tumor and Metastatics into one type
# combining dataframes into one
tumor  = df[df['sample_type'].isin(['Primary Tumor','Metastatic'])].copy()
normal = df[df['sample_type'] == 'Solid Tissue Normal'].copy()
combined = pd.concat([tumor, normal], ignore_index=True)
combined['label'] = (combined['sample_type'].isin(['Primary Tumor','Metastatic'])).astype(int)
print(f'Tumor samples:  {len(tumor)}')
print(f'Normal samples: {len(normal)}')

Tumor samples:  1113
Normal samples: 113


In [7]:
#showing combined dataframe head
print(combined.shape)
combined.head()

(1226, 15)


,sample,samples,sample_type,disease_type,primary_diagnosis,primary_site,disease_type.1,PGR,ERBB2,SLAMF7,PTEN,EGFR,CD276,ICAM1,label
0,TCGA-AC-A6IX-06A,TCGA-AC-A6IX-06A,Metastatic,Ductal and Lobular Neoplasms,"Lobular carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,0.2133,4.851,2.016,3.410,1.5460,4.185,3.049,1
1,TCGA-E2-A15E-06A,TCGA-E2-A15E-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,5.9380,7.051,2.208,4.504,0.6054,4.580,2.958,1
2,TCGA-E2-A15A-06A,TCGA-E2-A15A-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,2.8910,5.271,4.435,2.536,0.9872,4.173,3.025,1
3,TCGA-BH-A1FE-06A,TCGA-BH-A1FE-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,2.8380,3.028,2.612,3.743,3.2020,5.287,5.570,1
4,TCGA-BH-A1ES-06A,TCGA-BH-A1ES-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,2.5050,5.889,2.560,3.898,1.6520,4.293,4.300,1


# 3. Finding pairs and trios 

## Using gene-gene correlation
For pairs, we rank pairs of genes by their correlation score and return the top 5 scoring pairs. For the trio of genes, we compute every pairwise corrrelation and find the average of the of all three correlations. We then sort the list of all trio combinations and return the top 5 trios. 

In [8]:
#defining gene markers
features = ['PGR', 'ERBB2', 'SLAMF7', 'PTEN', 'EGFR', 'CD276', 'ICAM1']

#compute correlation matrix
corr_matrix = combined[features].corr(method='spearman')

#finding top pairs
pair_scores = []
for g1, g2 in itertools.combinations(features, 2):
    score = corr_matrix.loc[g1, g2]
    pair_scores.append((g1, g2, round(score, 4)))
    
#sorting all pair_scores and getting the top five
top_pairs = sorted(pair_scores, key=lambda x: x[2], reverse=True)[:5]

#printing results
print("\n=== TOP 5 PAIRS (highest co-expression) ===")
for g1, g2, score in top_pairs:
    print(f"{g1} + {g2}: {score}")

#finding top trios
trio_scores = []
for g1, g2, g3 in itertools.combinations(features, 3):
    s12 = corr_matrix.loc[g1, g2]
    s13 = corr_matrix.loc[g1, g3]
    s23 = corr_matrix.loc[g2, g3]
    avg_score = round((s12 + s13 + s23) / 3, 4)
    trio_scores.append((g1, g2, g3, avg_score))
    
#sorting all trio_scores and getting the top five
top_trios = sorted(trio_scores, key=lambda x: x[3], reverse=True)[:5]

#printing results
print("\n=== TOP 5 TRIOS (by average pairwise co-expression) ===")
for g1, g2, g3, score in top_trios:
    print(f"{g1} + {g2} + {g3}: {score}")
    


=== TOP 5 PAIRS (highest co-expression) ===
EGFR + ICAM1: 0.3489
PGR + PTEN: 0.3012
SLAMF7 + CD276: 0.2686
PTEN + EGFR: 0.2351
CD276 + ICAM1: 0.2197

=== TOP 5 TRIOS (by average pairwise co-expression) ===
EGFR + CD276 + ICAM1: 0.1856
PTEN + EGFR + ICAM1: 0.1848
SLAMF7 + CD276 + ICAM1: 0.1835
ERBB2 + SLAMF7 + CD276: 0.1301
PGR + ERBB2 + PTEN: 0.1033


## Using an Autoencoder model
We train an Autoencoder to learn to compress the input data and then reconstruct the output as closely to its original input. During the encoding phase, the input data get compressed into a latent space which we will use to compare gene coexpression. In the low-dimensional latent space it is easier to compare how close two vectors are;In this case, each vector encodes a single gene expression. 

In [9]:
#transposing features for normalization
expr = combined[features].T

#scaling data
scaler = StandardScaler()
expr_norm = scaler.fit_transform(expr.T).T
expr_norm = pd.DataFrame(expr_norm, index=features)
expr_norm = pd.DataFrame(expr_norm, index=features)

###  Describing Autoencoder network

In [10]:
# match transposed data dimension for input
input_dim = expr.shape[1]

#the compressed lower dimension representation created by the encoder
latent_dim = 16 

inputs = keras.Input(shape=(input_dim,))

# Encoder
encoded = layers.Dense(128, activation='relu')(inputs)
encoded = layers.Dropout(0.2)(encoded)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(latent_dim, name='embedding_layer')(encoded)  
# Decoder
decoded = layers.Dense(64, activation='relu')(encoded)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(input_dim, name='reconstruction')(decoded)

#declare full Autoencoder for training
autoencoder = keras.Model(inputs, decoded, name="Gene_Autoencoder")

#declare encoder for our embedding 
encoder = keras.Model(inputs, encoded, name="Gene_Encoder")   


autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "Gene_Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1226)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       157,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Dense)         │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction (Dense)          │ (None, 1226)           │       158,154 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,914 (1.27 MB)

 Trainable params: 333,914 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

### Training
We utilize early stopping to prevent overfitting, since we are only using 7 genes in total. The Early stopping monitors the model's loss and waits 50 epochs for the model's performance to improve before stopping training. 

In [11]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='loss', patience=50, restore_best_weights=True
)

history = autoencoder.fit(
    expr_norm, expr_norm,          # input = target (reconstruction)
    epochs=500,
    batch_size=7,                  # full batch on 7 genes
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

print(f"Training finished after {len(history.history['loss'])} epochs")

Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 753ms/step - loss: 1.0183
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 1.0052
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.9926
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.9838
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.9714
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.9508
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.9298
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.8962
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.8617
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.8254
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.7939
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.7498
Epoch 13/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.7137
Epoch 14/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.6694
Epoch 15/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.6290
Epoch 16/500
1/1 ━

### Extract Embeddings and Compute Co-Expression
We use the trained encoder to extract a lower-dimensional representation of the normalized gene expression data. Then we utilize cosine_similarity() to compute how closely two genes are from each other. This is analogues of a corrleation matrix but using the non-linear patterns that our encoder learned.

In [12]:
#Extracting embeddings from normalized gene expression data
embeddings = encoder.predict(expr_norm, verbose=0) 

# Cosine similarity matrix, aka learned co-expression
sim_matrix = cosine_similarity(embeddings)

### Ranking pairs and trio of genes

In [13]:
# === 5. TOP 5 PAIRS & TOP 5 TRIOS ===
pair_scores = []
for i, j in itertools.combinations(range(len(features)), 2):
    score = sim_matrix[i, j]
    pair_scores.append((features[i], features[j], round(float(score), 4)))

top_pairs = sorted(pair_scores, key=lambda x: x[2], reverse=True)[:5]

print("\n=== TOP 5 PAIRS (Keras AE cosine co-expression) ===")
for g1, g2, score in top_pairs:
    print(f"{g1} + {g2}: {score}")

trio_scores = []
for i, j, k in itertools.combinations(range(len(features)), 3):
    s12 = sim_matrix[i, j]
    s13 = sim_matrix[i, k]
    s23 = sim_matrix[j, k]
    avg_score = round((s12 + s13 + s23) / 3, 4)
    trio_scores.append((features[i], features[j], features[k], avg_score))

top_trios = sorted(trio_scores, key=lambda x: x[3], reverse=True)[:5]

print("\n=== TOP 5 TRIOS (by average cosine) ===")
for g1, g2, g3, score in top_trios:
    print(f"{g1} + {g2} + {g3}: {score} (avg)")


=== TOP 5 PAIRS (Keras AE cosine co-expression) ===
ERBB2 + CD276: 0.4493
SLAMF7 + CD276: 0.4176
PGR + PTEN: 0.3875
EGFR + ICAM1: 0.3663
CD276 + ICAM1: 0.354

=== TOP 5 TRIOS (by average cosine) ===
ERBB2 + SLAMF7 + CD276: 0.3898000121116638 (avg)
SLAMF7 + CD276 + ICAM1: 0.2870999872684479 (avg)
PGR + ERBB2 + CD276: 0.21490000188350677 (avg)
ERBB2 + CD276 + ICAM1: 0.18160000443458557 (avg)
PTEN + EGFR + ICAM1: 0.17000000178813934 (avg)
